# Survey Property Maps — All Bands × All DDF Fields — DP2 (USDF)

This notebook retrieves a selected **survey property map** (HealSparseMap) from the DP2 butler,
and displays a **2×3 subplot grid** (one panel per photometric band: u, g, r, i, z, y)
centred on each of the main LSST Deep Drilling Fields (DDF):

- **COSMOS**   — RA = 150.12°, Dec = +2.21°
- **ECDFS**    — RA =  53.13°, Dec = −28.10°
- **EDFS-a**   — RA =  58.90°, Dec = −49.32°
- **ELAIS-S1** — RA =   9.45°, Dec = −44.00°
- **XMM-LSS**  — RA =  35.71°, Dec =  −4.75°
- **ECDFS-ext** (ECDFS wide) — RA = 53.13°, Dec = −28.10° (same centre, wider span)

**Prerequisite**: the butler must be connected and `found_in` / `coll_hsp` must be set
(run the collection-search cells from `2026-03-11_ExploreDP2_SurveyPropertyMaps.ipynb` first,
or set `coll_hsp` manually below).

- **Author:** Sylvie Dagoret-Campagne  
- **Creation date:** 2026-03-11  
- **Environment:** USDF RSP — `LSST` kernel (`lsst_distrib` stack)  
- **Reference notebook:** `2026-03-11_ExploreDP2_SurveyPropertyMaps.ipynb`

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd

from lsst.daf.butler import Butler

## 2. Configuration

In [ ]:
# ── Butler repository and collection ──────────────────────────────────────────
REPO     = 'dp2_prep'
# Collection that contains the raw HealSparseMap objects
# (identified by running the search cells in the explorer notebook)
COLLECTION = 'LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545'

SKYMAP_NAME = 'lsst_cells_v2'

# ── Photometric bands (subplot order: row 0 = u g r, row 1 = i z y) ───────────
BANDS = ['u', 'g', 'r', 'i', 'z', 'y']

# ── Survey property map to display ────────────────────────────────────────────
# Choose one from the list below (comment/uncomment):
MAP_NAME = 'deepCoadd_psf_maglim_consolidated_map_weighted_mean'
#MAP_NAME = 'deepCoadd_psf_size_consolidated_map_weighted_mean'
#MAP_NAME = 'deepCoadd_exposure_time_consolidated_map_sum'
#MAP_NAME = 'deepCoadd_sky_background_consolidated_map_weighted_mean'
#MAP_NAME = 'deepCoadd_sky_noise_consolidated_map_weighted_mean'
#MAP_NAME = 'deepCoadd_epoch_consolidated_map_mean'

# ── Angular half-size of the displayed region around each field centre ────────
SPAN_DEG = 2.0    # degrees in declination

# ── Grid resolution for get_values_pos ────────────────────────────────────────
NPIX = 250

# ── Colormap ──────────────────────────────────────────────────────────────────
CMAP = 'viridis'

# ── Sentinel position (outside any DP2 coverage) ─────────────────────────────
SENTINEL_RA  = 180.0
SENTINEL_DEC = +60.0

In [ ]:
# ── LSST Deep Drilling Fields ─────────────────────────────────────────────────
# Each entry: (label, RA_deg, Dec_deg)
DDF_FIELDS = {
    'COSMOS'   : (150.12,  +2.21),
    'ECDFS'    : ( 53.13, -28.10),
    'EDFS-a'   : ( 58.90, -49.32),
    'ELAIS-S1' : (  9.45, -44.00),
    'XMM-LSS'  : ( 35.71,  -4.75),
}

## 3. Connect to the butler

In [ ]:
butler   = Butler(REPO)
registry = butler.registry
print(f"Butler connected to repository: {REPO}")
print(f"HealSparseMap collection      : {COLLECTION}")
print(f"Map                           : {MAP_NAME}")

## 4. Core helper functions

In [ ]:
def get_hspmap(map_name, band, skymap=SKYMAP_NAME, collection=COLLECTION):
    """
    Retrieve a HealSparseMap object from the butler.

    Parameters
    ----------
    map_name   : str   — butler dataset type name
    band       : str   — photometric band (e.g. 'r')
    skymap     : str   — skymap name
    collection : str   — butler collection

    Returns
    -------
    hspmap or None
    """
    try:
        return butler.get(
            map_name,
            dataId={'band': band, 'skymap': skymap},
            collections=collection
        )
    except Exception as e:
        print(f"  [{band}] retrieval failed: {type(e).__name__}: {e}")
        return None


def extract_region(hspmap, ra_cen, dec_cen, span_deg=SPAN_DEG, npix=NPIX):
    """
    Sample a HealSparseMap over a square region centred on (ra_cen, dec_cen).

    Returns x (RA grid), y (Dec grid), values (2-D array with sentinel -> NaN).
    """
    span_ra = span_deg / np.cos(np.deg2rad(dec_cen))
    ra      = np.linspace(ra_cen  - span_ra,  ra_cen  + span_ra,  npix)
    dec     = np.linspace(dec_cen - span_deg, dec_cen + span_deg, npix)
    x, y   = np.meshgrid(ra, dec)

    values   = hspmap.get_values_pos(x, y).astype(float)
    sentinel = float(hspmap.get_values_pos(SENTINEL_RA, SENTINEL_DEC))
    values[values == sentinel] = np.nan

    return x, y, values


def stats_str(values):
    """Return a compact statistics string for a 2-D array (NaN-safe)."""
    return (
        f"min={np.nanmin(values):.3f}  "
        f"max={np.nanmax(values):.3f}  "
        f"med={np.nanmedian(values):.3f}  "
        f"std={np.nanstd(values):.3f}"
    )

print("Helper functions defined.")

## 5. Display function: 2×3 subplot grid for one field

For a given DDF field and a given map, retrieve the HealSparseMap for each of the
6 bands and display them in a 2-row × 3-column grid with a shared colour scale.

In [ ]:
def plot_field_all_bands(
    field_name,
    ra_cen,
    dec_cen,
    map_name=MAP_NAME,
    bands=BANDS,
    span_deg=SPAN_DEG,
    npix=NPIX,
    cmap=CMAP,
    share_scale=True,
    figsize=(18, 10),
):
    """
    Plot a 2x3 grid of survey property map panels, one per photometric band,
    centred on a DDF field.

    Parameters
    ----------
    field_name  : str   — label shown in titles (e.g. 'COSMOS')
    ra_cen      : float — field centre RA (deg)
    dec_cen     : float — field centre Dec (deg)
    map_name    : str   — butler dataset type name for the HealSparseMap
    bands       : list  — photometric bands to display
    span_deg    : float — half-size of the displayed region in Dec (deg)
    npix        : int   — sampling grid resolution
    cmap        : str   — matplotlib colormap
    share_scale : bool  — if True, use a common colour scale across all bands
    figsize     : tuple — figure size in inches
    """
    print(f"\n{'='*70}")
    print(f"Field: {field_name}   RA={ra_cen:.4f}  Dec={dec_cen:.4f}")
    print(f"Map  : {map_name}")
    print(f"{'='*70}")

    # ── Retrieve all maps and extract regions ──────────────────────────────────
    data = {}    # band -> (x, y, values)
    for band in bands:
        hsp = get_hspmap(map_name, band)
        if hsp is None:
            data[band] = None
            continue
        x, y, values = extract_region(hsp, ra_cen, dec_cen, span_deg, npix)
        data[band] = (x, y, values)
        print(f"  band={band}  {stats_str(values)}")

    # ── Compute common colour scale ────────────────────────────────────────────
    if share_scale:
        all_vals = np.concatenate([
            v[2].ravel() for v in data.values()
            if v is not None
        ])
        vmin = np.nanpercentile(all_vals, 2)
        vmax = np.nanpercentile(all_vals, 98)
    else:
        vmin = vmax = None    # per-panel auto-scale

    # ── Build 2x3 figure ──────────────────────────────────────────────────────
    nrows, ncols = 2, 3
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=figsize,
        layout='constrained'
    )

    short_map = (
        map_name
        .replace('deepCoadd_', '')
        .replace('_consolidated_map_', ' ')
    )
    fig.suptitle(
        f"{short_map}\nField: {field_name}  "
        f"(RA={ra_cen:.3f}°, Dec={dec_cen:.3f}°)  |  "
        f"span ±{span_deg}° in Dec",
        fontsize=13, fontweight='bold'
    )

    for idx, band in enumerate(bands):
        ax = axes[idx // ncols, idx % ncols]

        if data[band] is None:
            ax.text(0.5, 0.5, f'band {band}\nnot available',
                    ha='center', va='center', transform=ax.transAxes,
                    fontsize=12, color='red')
            ax.set_title(f"band = {band}", fontweight='bold')
            ax.axis('off')
            continue

        x, y, values = data[band]

        # Per-panel scale if share_scale is False
        _vmin = vmin if share_scale else np.nanpercentile(values, 2)
        _vmax = vmax if share_scale else np.nanpercentile(values, 98)

        pcm = ax.pcolormesh(x, y, values, cmap=cmap, vmin=_vmin, vmax=_vmax)
        fig.colorbar(pcm, ax=ax, shrink=0.85, pad=0.02)

        # Mark the field centre
        ax.plot(ra_cen, dec_cen, '+', color='red', ms=10, mew=2,
                label=f'{field_name} centre')

        ax.set_title(f"band = {band}", fontsize=12, fontweight='bold')
        ax.set_xlabel("RA (deg)", fontsize=9)
        ax.set_ylabel("Dec (deg)", fontsize=9)
        ax.invert_xaxis()    # East to the left (astronomical convention)

        # Compact statistics in the panel
        stats = stats_str(values)
        ax.set_xlabel(f"RA (deg)\n{stats}", fontsize=8)

    plt.show()

print("plot_field_all_bands() defined.")

## 6. COSMOS — 2×3 grid, all bands

In [ ]:
FIELD = 'COSMOS'
RA_CEN, DEC_CEN = DDF_FIELDS[FIELD]

plot_field_all_bands(FIELD, RA_CEN, DEC_CEN)

## 7. ECDFS — 2×3 grid, all bands

In [ ]:
FIELD = 'ECDFS'
RA_CEN, DEC_CEN = DDF_FIELDS[FIELD]

plot_field_all_bands(FIELD, RA_CEN, DEC_CEN)

## 8. EDFS-a — 2×3 grid, all bands

In [ ]:
FIELD = 'EDFS-a'
RA_CEN, DEC_CEN = DDF_FIELDS[FIELD]

plot_field_all_bands(FIELD, RA_CEN, DEC_CEN)

## 9. ELAIS-S1 — 2×3 grid, all bands

In [ ]:
FIELD = 'ELAIS-S1'
RA_CEN, DEC_CEN = DDF_FIELDS[FIELD]

plot_field_all_bands(FIELD, RA_CEN, DEC_CEN)

## 10. XMM-LSS — 2×3 grid, all bands

In [ ]:
FIELD = 'XMM-LSS'
RA_CEN, DEC_CEN = DDF_FIELDS[FIELD]

plot_field_all_bands(FIELD, RA_CEN, DEC_CEN)

## 11. Full loop — all DDF fields, all bands

Iterate automatically over all fields defined in `DDF_FIELDS`.

In [ ]:
for field_name, (ra_cen, dec_cen) in DDF_FIELDS.items():
    plot_field_all_bands(field_name, ra_cen, dec_cen)

## 12. Loop over multiple maps for a single field

Display all available survey property maps for COSMOS.

In [ ]:
# All raw HealSparseMap names (derived from Plot names by stripping prefix/suffix)
ALL_MAPS = [
    'deepCoadd_dcr_ddec_consolidated_map_weighted_mean',
    'deepCoadd_dcr_dra_consolidated_map_weighted_mean',
    'deepCoadd_dcr_e1_consolidated_map_weighted_mean',
    'deepCoadd_dcr_e2_consolidated_map_weighted_mean',
    'deepCoadd_epoch_consolidated_map_max',
    'deepCoadd_epoch_consolidated_map_mean',
    'deepCoadd_epoch_consolidated_map_min',
    'deepCoadd_exposure_time_consolidated_map_sum',
    'deepCoadd_psf_e1_consolidated_map_weighted_mean',
    'deepCoadd_psf_e2_consolidated_map_weighted_mean',
    'deepCoadd_psf_maglim_consolidated_map_weighted_mean',
    'deepCoadd_psf_size_consolidated_map_weighted_mean',
    'deepCoadd_sky_background_consolidated_map_weighted_mean',
    'deepCoadd_sky_noise_consolidated_map_weighted_mean',
]

FIELD  = 'COSMOS'
RA_CEN, DEC_CEN = DDF_FIELDS[FIELD]

for map_name in ALL_MAPS:
    plot_field_all_bands(FIELD, RA_CEN, DEC_CEN, map_name=map_name)

## 13. Full exploration: all maps × all fields

⚠️ This cell produces 14 maps × 5 fields = **70 figures**.  
Uncomment and run only if needed.

In [ ]:
# for map_name in ALL_MAPS:
#     for field_name, (ra_cen, dec_cen) in DDF_FIELDS.items():
#         plot_field_all_bands(field_name, ra_cen, dec_cen, map_name=map_name)

## 14. Variant: per-band colour scale (share_scale=False)

Useful when the dynamic range between bands is very different  
(e.g. exposure time or epoch maps).

In [ ]:
FIELD = 'COSMOS'
RA_CEN, DEC_CEN = DDF_FIELDS[FIELD]

plot_field_all_bands(
    FIELD, RA_CEN, DEC_CEN,
    map_name='deepCoadd_exposure_time_consolidated_map_sum',
    share_scale=False
)